#**0. Imports**

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.datasets as dset
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Subset, DataLoader, ConcatDataset
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

#**1. CIFAR-10 + ResNet Feature Extractor (UNCHANGED)**

In [2]:
transform_resnet = T.Compose([
    T.ToTensor(),
    T.Resize((224,224), antialias=True),
    T.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
])

full_train_dataset = dset.CIFAR10(
    root="./data", train=True, download=True, transform=transform_resnet
)

all_labels = np.array(full_train_dataset.targets)

resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Identity()
resnet.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = resnet.to(device)

100%|██████████| 170M/170M [00:01<00:00, 86.9MB/s] 


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


#**2. Feature Extraction (UNCHANGED)**

In [3]:
def extract_features_batched(dataset, batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    feats = []

    with torch.no_grad():
        for x,_ in loader:
            x = x.to(device)
            f = resnet(x)
            feats.append(f.cpu().numpy())

    return np.concatenate(feats, axis=0)

#**3. Imbalanced Data Distribution (WITHOUT replacement)**

In [4]:
def generate_imbalanced_split(total, n_clients, alpha=0.5):
    """
    Generate an imbalanced split of `total` samples across n_clients.
    Lower alpha → more imbalance.
    """
    # Dirichlet gives proportions that sum to 1
    proportions = np.random.dirichlet(alpha=[alpha]*n_clients)
    #print(f"proportions: {proportions}")
    # Convert to counts
    raw_counts = (proportions * total).astype(int)
    #print(f"raw_counts: {raw_counts}")
    # Fix rounding issues
    diff = total - raw_counts.sum()
    raw_counts[np.argmax(raw_counts)] += diff
    #print(f"final_proportions: {proportions}")
    return raw_counts

In [5]:
def rotate_list(lst, k):
    return lst[k:] + lst[:k]

In [6]:
def distribute_cifar_imbalanced(labels, n_clients, alpha=0.5):
    """
    Distribute CIFAR-10 data across n_clients with class-wise imbalance.
    Returns dict: client_id → list of sample indices
    """

    client_indices = {i: [] for i in range(n_clients)}

    for cls in range(10):
        cls_idx = np.where(labels == cls)[0]
        np.random.shuffle(cls_idx)

        # Generate imbalance for this class
        base_split = generate_imbalanced_split(
            total=len(cls_idx),
            n_clients=n_clients,
            alpha=alpha
        )

        # Rotate per class (important!)
        split = rotate_list(list(base_split), cls % n_clients)
        print(f"cls: {cls} base_split: {base_split} split_after_rotate: {split}")

        start = 0
        for cid in range(n_clients):
            count = split[cid]
            client_indices[cid].extend(
                cls_idx[start:start+count]
            )
            start += count

        assert start == len(cls_idx), "Class split mismatch!"

    return client_indices

#**4. Initialize Clients**

In [7]:
N_CLIENTS = 3

client_indices = distribute_cifar_imbalanced(
    labels=all_labels,
    n_clients=N_CLIENTS,
    alpha=0.5
)


clients = {}
for cid in range(N_CLIENTS):
    Xc = Subset(full_train_dataset, client_indices[cid])
    yc = all_labels[client_indices[cid]]

    clients[cid] = {"X": Xc, "y": yc}

    print(f"Client {cid} class dist:", Counter(yc))

cls: 0 base_split: [  68  495 4437] split_after_rotate: [np.int64(68), np.int64(495), np.int64(4437)]
cls: 1 base_split: [2639 1908  453] split_after_rotate: [np.int64(1908), np.int64(453), np.int64(2639)]
cls: 2 base_split: [3633 1345   22] split_after_rotate: [np.int64(22), np.int64(3633), np.int64(1345)]
cls: 3 base_split: [2440 2558    2] split_after_rotate: [np.int64(2440), np.int64(2558), np.int64(2)]
cls: 4 base_split: [ 939 1285 2776] split_after_rotate: [np.int64(1285), np.int64(2776), np.int64(939)]
cls: 5 base_split: [3477 1106  417] split_after_rotate: [np.int64(417), np.int64(3477), np.int64(1106)]
cls: 6 base_split: [ 243 4180  577] split_after_rotate: [np.int64(243), np.int64(4180), np.int64(577)]
cls: 7 base_split: [3257 1675   68] split_after_rotate: [np.int64(1675), np.int64(68), np.int64(3257)]
cls: 8 base_split: [   5 4791  204] split_after_rotate: [np.int64(204), np.int64(5), np.int64(4791)]
cls: 9 base_split: [1863 2049 1088] split_after_rotate: [np.int64(1863), n

#**5. Local & Global Distributions and Normalization**

In [8]:
def compute_LD(y, n_classes=10):
    """
    Returns raw class-count vector for a client
    LD = [C0, C1, ..., C9]
    """
    return np.bincount(y, minlength=n_classes)

def compute_GD(LDs):
    """
    LDs: dict {cid -> raw LD vector}
    GD = sum of all LDs
    """
    return np.sum(list(LDs.values()), axis=0)

def normalize(dist):
    """
    Converts count vector to probability distribution
    """
    total = dist.sum()
    if total == 0:
        return dist
    return dist / total

#**6. Dominant Client Selection**

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def find_dominant_client_with_saturation(LDs, GD, clients, excluded_clients):
    sims = {}

    for cid, LD in LDs.items():
        if cid in excluded_clients:
            continue

        sims[cid] = cosine_similarity(
            LD.reshape(1, -1),
            GD.reshape(1, -1)
        )[0, 0]

    if len(sims) == 0:
        return None, {}

    dom = max(sims, key=sims.get)
    return dom, sims

#**7. Flicker Oversampling (UNCHANGED LOGIC)**

In [10]:
def flicker_oversample(X_client, y_client, lin_thres=0.5):
    class_counts = Counter(y_client)
    Lmax = max(class_counts.values())

    new_X = [X_client]
    new_y = list(y_client)

    for cls, cnt in class_counts.items():
        Lin = cnt / Lmax
        if Lin < lin_thres:
            N_add = int(np.ceil(1 / Lin))
            cls_idx = np.where(y_client == cls)[0]
            chosen = np.random.choice(cls_idx, N_add, replace=True)

            base_ds = dset.CIFAR10(
                root="./data", train=True, download=False, transform=None
            )

            aug = T.Compose([
                T.RandomHorizontalFlip(),
                T.RandomRotation(10),
                T.ToTensor(),
                T.Resize((224,224)),
                T.Normalize([0.485,0.456,0.406],
                            [0.229,0.224,0.225])
            ])

            imgs = []
            labels = []

            for idx in chosen:
                img,_ = base_ds[X_client.indices[idx]]
                imgs.append(aug(img))
                labels.append(cls)

            new_X.append(torch.utils.data.TensorDataset(
                torch.stack(imgs),
                torch.tensor(labels)
            ))
            new_y.extend(labels)

    return ConcatDataset(new_X), np.array(new_y)

#**8. Flicker Undersampling – LOCAL Redundancy (UNCHANGED)**

In [11]:
def get_majority_classes(y, lin_threshold=0.9, n_classes=10):
    cnt = np.bincount(y, minlength=n_classes)
    Lmax = cnt.max()

    majority_classes = [
        c for c in range(n_classes)
        if cnt[c] / Lmax >= lin_threshold
    ]

    return majority_classes

In [12]:
def local_redundancy_majority(X, y, theta, lin_threshold=0.9):
    maj_classes = get_majority_classes(y, lin_threshold)

    if len(maj_classes) == 0:
        return np.array([]), np.array([])

    maj_idx = np.where(np.isin(y, maj_classes))[0]
    X_maj = Subset(X, maj_idx)

    feats = extract_features_batched(X_maj)
    feats = feats / np.linalg.norm(feats, axis=1, keepdims=True)

    sim = feats @ feats.T
    mean_sim = sim.mean(axis=1)
    var_sim = sim.var(axis=1)

    k = max(1, int(theta * len(mean_sim)))
    buffer_local = np.argsort(mean_sim)[-k:]
    buffer_local = buffer_local[np.argsort(var_sim[buffer_local])[::-1]]

    buffer_idx = maj_idx[buffer_local]

    return buffer_idx, feats[buffer_local]

#**9. Flicker Undersampling – GLOBAL Redundancy (NEW)**

In [13]:
def flicker_undersample_global(
    dom_id, clients, theta=0.2, eta=0.6, lin_threshold=0.9
):
    Xd = clients[dom_id]["X"]
    yd = clients[dom_id]["y"]

    # Step 1: Local redundancy (majority-only)
    buffer_idx, buffer_vecs = local_redundancy_majority(
        Xd, yd, theta, lin_threshold
    )

    if len(buffer_idx) == 0:
        return Xd, yd

    # Step 2: Extract features of other clients (per client)
    other_client_feats = {}

    for cid, client in clients.items():
        if cid == dom_id:
            continue

        feats = extract_features_batched(client["X"])
        feats = feats / np.linalg.norm(feats, axis=1, keepdims=True)
        other_client_feats[cid] = feats

    # Step 3: Global redundancy check (CLIENT-AVERAGED)
    remove = []

    for i, idx in enumerate(buffer_idx):
        client_avgs = []

        for feats in other_client_feats.values():
            sims = cosine_similarity(
                buffer_vecs[i].reshape(1, -1),
                feats
            )[0]   # shape: (num_samples_client,)

            client_avgs.append(np.mean(sims))

        final_avg = np.mean(client_avgs)

        if final_avg >= eta:
            remove.append(idx)

    # Step 4: Remove redundant samples
    mask = np.ones(len(Xd), dtype=bool)
    mask[remove] = False

    return Subset(Xd, np.where(mask)[0]), yd[mask]

#**10. Full Training Simulation Loop**

**Defining Max_rounds, Maximum_local_imbalance for each clients**

In [14]:
MAX_DOM_ROUNDS = 6
LIn_MAX = 0.75   # between 0.70–0.80 as you said

dominance_count = {cid: 0 for cid in clients}
excluded_clients = set()
# 0 → next time oversample
# 1 → next time undersample
client_resample_flag = {cid: 0 for cid in clients}

In [15]:
def compute_LIn(y):
    cnt = np.bincount(y, minlength=10)
    return cnt.min() / cnt.max()

In [16]:
ROUNDS = 25   # global loop; dominance is client-limited

for r in range(ROUNDS):
    print(f"\n------ ROUND {r} ----------")

    # Step 1: Compute LDs (raw counts normalized for cosine)
    LDs = {
        cid: np.bincount(clients[cid]["y"], minlength=10)
        for cid in clients
    }

    # Normalize LDs for cosine similarity
    LDs_norm = {
        cid: LDs[cid] / np.linalg.norm(LDs[cid])
        for cid in LDs
    }

    # Step 2: Compute GD
    GD = sum(LDs.values())
    GD_norm = GD / np.linalg.norm(GD)

    # Step 3: Find dominant client (with saturation)
    dom, sims = find_dominant_client_with_saturation(
        LDs_norm, GD_norm, clients, excluded_clients
    )

    if dom is None:
        print("No eligible dominant clients left. Stopping.")
        break

    print("Dominant client:", dom)

    # Step 4: Check saturation conditions
    Lin_dom = compute_LIn(clients[dom]["y"])

    if dominance_count[dom] >= MAX_DOM_ROUNDS or Lin_dom >= LIn_MAX:
        print(f"Client {dom} saturated (M={dominance_count[dom]}, LIn={Lin_dom:.3f})")
        excluded_clients.add(dom)
        continue   # reselect dominant in next round

    # Step 5: Oversample / Undersample (client-specific toggle)
    if client_resample_flag[dom] == 0:
        print("→ Oversampling")
        Xn, yn = flicker_oversample(
            clients[dom]["X"],
            clients[dom]["y"]
        )
        client_resample_flag[dom] = 1
    else:
        print("→ Undersampling (Local + Global)")
        Xn, yn = flicker_undersample_global(dom, clients)
        client_resample_flag[dom] = 0

    # Step 6: Update client
    clients[dom]["X"] = Xn
    clients[dom]["y"] = yn
    dominance_count[dom] += 1

    print("Updated dist:", Counter(yn))


------ ROUND 0 ----------
Dominant client: 1
→ Oversampling
Updated dist: Counter({np.int64(6): 4180, np.int64(2): 3633, np.int64(5): 3477, np.int64(4): 2776, np.int64(3): 2558, np.int64(9): 2052, np.int64(8): 841, np.int64(0): 504, np.int64(1): 463, np.int64(7): 130})

------ ROUND 1 ----------
Dominant client: 1
→ Undersampling (Local + Global)
Updated dist: Counter({np.int64(2): 3633, np.int64(5): 3477, np.int64(6): 3344, np.int64(4): 2776, np.int64(3): 2558, np.int64(9): 2052, np.int64(8): 841, np.int64(0): 504, np.int64(1): 463, np.int64(7): 130})

------ ROUND 2 ----------
Dominant client: 1
→ Oversampling
Updated dist: Counter({np.int64(2): 3633, np.int64(5): 3477, np.int64(6): 3344, np.int64(4): 2776, np.int64(3): 2558, np.int64(9): 2052, np.int64(8): 846, np.int64(0): 512, np.int64(1): 471, np.int64(7): 158})

------ ROUND 3 ----------
Dominant client: 1
→ Undersampling (Local + Global)
Updated dist: Counter({np.int64(2): 2799, np.int64(6): 2793, np.int64(4): 2776, np.int64(5

#**Training and Testing Phase**

**Merge the dataset as of now**

In [17]:
from torch.utils.data import ConcatDataset

final_train_dataset = ConcatDataset(
    [clients[cid]["X"] for cid in clients]
)

print("Final training size:", len(final_train_dataset))

Final training size: 41472


In [18]:
final_labels = np.concatenate(
    [clients[cid]["y"] for cid in clients]
)

print("Final class distribution:", Counter(final_labels))

Final class distribution: Counter({np.int64(3): 5578, np.int64(1): 4472, np.int64(9): 4425, np.int64(2): 4111, np.int64(5): 4062, np.int64(7): 3976, np.int64(8): 3877, np.int64(4): 3801, np.int64(0): 3630, np.int64(6): 3540})


In [19]:
def safe_collate(batch):
    xs, ys = zip(*batch)

    xs = torch.stack(xs)
    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys

#Load test Data

In [20]:
test_dataset = dset.CIFAR10(
    root="./data", train=False, download=True,
    transform=transform_resnet
)

test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False
)

In [21]:
train_loader = DataLoader(
    final_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=safe_collate
)

**Define the Model**

In [22]:
class CIFARClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.fc = nn.Identity()

        for p in self.backbone.parameters(): #freezing backbone
            p.requires_grad = False   # resnet is used only as a feature extractor(no weights)

        self.head = nn.Linear(512, 10) # only trainable part y = Wz + b, z = 512 features(10 outputs)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

In [23]:
model = CIFARClassifier().to(device)

In [24]:
criterion = nn.CrossEntropyLoss() #penalizes wrong class predictions
optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3) #only head parameters are updated

**Training Loop**

In [25]:
def train(model, loader, optimizer, criterion, epochs=10):
    for ep in range(epochs):   # 1 epoch <-- one full pass over training data(10 times)
        model.train()
        total_loss = 0

        for x, y in loader:
            x, y = x.to(device), y.to(device) # model and data are one same device

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y) # how wrong these predictions are
            loss.backward() #compute ogradients of classifier head
            optimizer.step() #update model head weights and bias

            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / len(loader.dataset)
        print(f"Epoch {ep}: train loss = {avg_loss:.4f}")

In [26]:
train(model, train_loader, optimizer, criterion, epochs=10)

Epoch 0: train loss = 0.9243
Epoch 1: train loss = 0.6196
Epoch 2: train loss = 0.5771
Epoch 3: train loss = 0.5577
Epoch 4: train loss = 0.5462
Epoch 5: train loss = 0.5363
Epoch 6: train loss = 0.5287
Epoch 7: train loss = 0.5237
Epoch 8: train loss = 0.5213
Epoch 9: train loss = 0.5166


**Evaluate the Model**

In [27]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [28]:
test_acc = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.8013


**Optional**

In [29]:
def per_class_accuracy(model, loader, n_classes=10):
    model.eval()
    correct = np.zeros(n_classes)
    total = np.zeros(n_classes)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)

            for i in range(len(y)):
                total[y[i]] += 1
                correct[y[i]] += (preds[i] == y[i]).item()

    return correct / np.maximum(total, 1)

In [30]:
print("Per-class accuracy:", per_class_accuracy(model, test_loader))

Per-class accuracy: [0.791 0.88  0.81  0.583 0.79  0.745 0.807 0.852 0.883 0.872]


In [1]:
#Hi Behera